# Notebook 03 - Calibration Model Development

**Milestone:** HV-006  
**Technical Gate:** CM-001  
**Primary Engineering Question:** Calibration Model Development

This notebook develops and freezes an RH calibration model using only qualified analyst-visible `STATIC / CALIBRATION` evidence from the frozen `VSM-LAB-DATA-v1.1` release.

The notebook must preserve the analyst-blind boundary established by DQ-001 and the calibration/validation separation required by repository governance.

**No VALIDATION measurement result is used for fitting, tuning, feature selection, or model selection in HV-006.**

## 1. Engineering Question

Can a calibration model built only from the frozen CALIBRATION subset reduce systematic RH measurement error while remaining:

- interpretable;
- free from experimental-unit leakage;
- operationally deployable; and
- suitable for later blind independent validation?

HV-006 answers the **model-development** question only.

The independent question—

> Does the frozen calibration generalize to reserved validation data?

—is deferred to Notebook 04.

## 2. Scientific Boundary

### Included

- `STATIC / CALIBRATION` measurement records only;
- public test-matrix design metadata;
- Test_ID-level calibration summaries;
- DUT-observable candidate predictors;
- calibration-only internal cross-validation;
- calibration-only residual diagnostics;
- final model freeze.

### Prohibited in HV-006

- VALIDATION measurement values or validation performance metrics;
- hidden simulator truth;
- private DUT personality/configuration;
- target-side/reference variables as deployable predictors;
- sample-level random train/test splitting;
- model choice informed by Notebook 04.

The public test matrix may be used to verify that a reserved validation design exists, but its measurement values remain outside the HV-006 evidence boundary.

## 3. Experimental Unit and Calibration Target

The primary model-development unit is **one summary record per CALIBRATION Test_ID**.

Each Test_ID contains 60 repeated 1 Hz measurements under one nominal static condition. Those 60 samples are useful for estimating the Test_ID mean and within-test scatter, but they are **not 60 independent calibration experiments**.

Therefore:

```text
5,760 CALIBRATION samples
        ↓ summarize by Test_ID
96 calibration modeling records
```

The pre-calibration error convention remains:

```text
Error_RH_Pre = RH_DUT_Mean - RH_Reference_Mean
```

The correction target is:

```text
Correction_RH = RH_Reference_Mean - RH_DUT_Mean
              = -Error_RH_Pre
```

Later, the calibrated result will be:

```text
RH_Calibrated = RH_DUT + Predicted_Correction_RH
```

and the post-calibration residual will use:

```text
Residual_Cal = RH_Calibrated - RH_Reference
```

The reference indication is a reference measurement, not exact physical truth.

## 4. Reproducible Setup

Reusable calibration loading and Test_ID-summary logic is kept in `src/calibration_analysis.py`.

This notebook remains the engineering record: it verifies the analysis boundary, shows the design, constructs the modeling table, and later documents model comparison and freeze decisions.

In [ ]:
from pathlib import Path
import sys

import numpy as np
import pandas as pd


# ------------------------------------------------------------
# Locate project root and import reusable analyst-side helpers
# ------------------------------------------------------------

current_dir = Path.cwd()
candidate_roots = [current_dir, *current_dir.parents]

project_root = next(
    (
        path
        for path in candidate_roots
        if (path / "data" / "raw" / "vsm_lab_data_v1_1").exists()
    ),
    None,
)

if project_root is None:
    raise FileNotFoundError(
        "Could not locate the project root containing "
        "'data/raw/vsm_lab_data_v1_1'."
    )

if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

from src.calibration_analysis import (
    calibration_design_summary,
    load_static_calibration,
    summarize_calibration_test_ids,
)

reports_dir = project_root / "reports"
figures_dir = reports_dir / "figures"
tables_dir = reports_dir / "tables"

figures_dir.mkdir(parents=True, exist_ok=True)
tables_dir.mkdir(parents=True, exist_ok=True)

print(f"Project root : {project_root}")
print(f"Figures dir : {figures_dir}")
print(f"Tables dir  : {tables_dir}")

## 5. CM-01 — CALIBRATION Data Boundary Audit

The first executable requirement is to prove that the working measurement frame contains only `STATIC / CALIBRATION` evidence.

Expected calibration design from the frozen public test matrix:

```text
4 RH levels × 3 temperatures × 2 directions × 2 days × 2 runs
= 96 CALIBRATION Test_IDs

96 Test_IDs × 60 samples/Test_ID
= 5,760 measurement rows
```

The reserved validation design is checked only through the public test matrix. No validation measurement values are summarized or evaluated here.

In [ ]:
# ------------------------------------------------------------
# CM-01A: Load calibration evidence and public design metadata
# ------------------------------------------------------------

calibration_df, test_matrix_df = load_static_calibration(project_root)

calibration_matrix_df = test_matrix_df.loc[
    (test_matrix_df["Test_Family"] == "STATIC")
    & (test_matrix_df["Analysis_Role"] == "CALIBRATION")
].copy()

validation_matrix_df = test_matrix_df.loc[
    (test_matrix_df["Test_Family"] == "STATIC")
    & (test_matrix_df["Analysis_Role"] == "VALIDATION")
].copy()

samples_per_test = calibration_df.groupby("Test_ID").size()

print(f"CALIBRATION measurement rows : {len(calibration_df):,}")
print(f"CALIBRATION Test_IDs          : {calibration_df['Test_ID'].nunique()}")
print(f"CALIBRATION matrix Test_IDs   : {calibration_matrix_df['Test_ID'].nunique()}")
print(f"Reserved VALIDATION Test_IDs  : {validation_matrix_df['Test_ID'].nunique()}")
print(
    "Samples/Test_ID values       : "
    f"{sorted(samples_per_test.unique().tolist())}"
)
print(
    "Working Analysis_Role values : "
    f"{sorted(calibration_df['Analysis_Role'].unique().tolist())}"
)

assert len(calibration_df) == 5760
assert calibration_df["Test_ID"].nunique() == 96
assert calibration_matrix_df["Test_ID"].nunique() == 96
assert validation_matrix_df["Test_ID"].nunique() == 36
assert set(samples_per_test.unique()) == {60}
assert set(calibration_df["Test_Family"].unique()) == {"STATIC"}
assert set(calibration_df["Analysis_Role"].unique()) == {"CALIBRATION"}

calibration_measurement_ids = set(calibration_df["Test_ID"])
calibration_matrix_ids = set(calibration_matrix_df["Test_ID"])
validation_matrix_ids = set(validation_matrix_df["Test_ID"])

assert calibration_measurement_ids == calibration_matrix_ids
assert calibration_measurement_ids.isdisjoint(validation_matrix_ids)

print("\nCM-01 boundary audit: PASS")

## 6. CM-02 — Test_ID-Level Calibration Design

Now the 5,760 repeated samples are reduced to one traceable record per Test_ID.

The expected modeling design is:

- RH setpoints: 20, 40, 60, 80 %RH;
- temperature setpoints: 15, 25, 35 °C;
- Direction: Up / Down;
- Day: 1 / 2;
- Run: 1 / 2;
- 24 unique physical-condition groups defined by RH × Temperature × Direction;
- four Day × Run repetitions per physical-condition group.

This structure will later support group-aware cross-validation without allowing repetitions of the same physical condition to leak across training and assessment folds.

In [ ]:
# ------------------------------------------------------------
# CM-02A: Build one calibration record per Test_ID
# ------------------------------------------------------------

calibration_summary_df = summarize_calibration_test_ids(calibration_df)
design = calibration_design_summary(calibration_summary_df)

print("Calibration design audit")
print("------------------------")
for key, value in design.items():
    print(f"{key:28s}: {value}")

assert len(calibration_summary_df) == 96
assert design["N_Test_IDs"] == 96
assert design["RH_Levels"] == [20.0, 40.0, 60.0, 80.0]
assert design["Temperature_Levels_C"] == [15.0, 25.0, 35.0]
assert design["Directions"] == ["Down", "Up"]
assert design["Days"] == [1, 2]
assert design["Runs"] == [1, 2]
assert design["N_Condition_Groups"] == 24
assert design["Samples_Per_Test_ID_Unique"] == [60]

condition_replication = (
    calibration_summary_df
    .groupby("Condition_ID")
    .size()
)

assert set(condition_replication.unique()) == {4}

display(
    calibration_summary_df[
        [
            "Test_ID",
            "RH_Setpoint",
            "Temperature_Setpoint_C",
            "Direction",
            "Day_ID",
            "Run_ID",
            "N_Samples",
            "RH_DUT_Mean",
            "RH_Reference_Mean",
            "Temperature_DUT_Mean_C",
            "Error_RH_Pre",
            "Correction_RH",
            "Condition_ID",
        ]
    ].head(12).round(4)
)

print("\nCM-02 Test_ID summary audit: PASS")

## 7. CM-03 — Calibration Target Verification

Before any model is created, verify the sign convention algebraically and numerically.

For every Test_ID:

```text
Error_RH_Pre + Correction_RH = 0
```

because:

```text
Error_RH_Pre = RH_DUT_Mean - RH_Reference_Mean
Correction_RH = RH_Reference_Mean - RH_DUT_Mean
```

This check prevents a sign error from propagating through the entire calibration workflow.

In [ ]:
# ------------------------------------------------------------
# CM-03A: Verify correction-target sign convention
# ------------------------------------------------------------

correction_identity_error = (
    calibration_summary_df["Correction_RH"]
    - (
        calibration_summary_df["RH_Reference_Mean"]
        - calibration_summary_df["RH_DUT_Mean"]
    )
)

sign_identity_error = (
    calibration_summary_df["Error_RH_Pre"]
    + calibration_summary_df["Correction_RH"]
)

assert np.allclose(correction_identity_error, 0.0)
assert np.allclose(sign_identity_error, 0.0)

print(
    "Maximum correction-definition discrepancy : "
    f"{correction_identity_error.abs().max():.3e} %RH"
)
print(
    "Maximum Error + Correction discrepancy     : "
    f"{sign_identity_error.abs().max():.3e} %RH"
)
print("\nCM-03 correction-target verification: PASS")

## 8. CM-04 — Modeling Table and Predictor-Leakage Audit

The deployable model-selection track starts with only DUT-observable predictors:

```text
RH_DUT_Mean
Temperature_DUT_Mean_C
```

The target is:

```text
Correction_RH
```

The following variables remain available for design control or diagnostics but are **not deployable predictors**:

- `RH_Reference_Mean`;
- `Temperature_Reference_Mean_C`;
- `RH_Setpoint`;
- `Temperature_Setpoint_C`;
- `Day_ID`;
- `Run_ID`;
- `Test_ID`.

`Direction` is retained as a diagnostic design factor because HV-004 showed substantial UP/DOWN structure. It is not automatically eligible for the final deployable model.

In [ ]:
# ------------------------------------------------------------
# CM-04A: Define deployable predictors, target, groups, diagnostics
# ------------------------------------------------------------

deployable_predictors = [
    "RH_DUT_Mean",
    "Temperature_DUT_Mean_C",
]

target_column = "Correction_RH"
group_column = "Condition_ID"

diagnostic_columns = [
    "RH_Setpoint",
    "Temperature_Setpoint_C",
    "Direction",
    "Day_ID",
    "Run_ID",
    "Error_RH_Pre",
]

prohibited_predictor_columns = {
    "RH_Reference_Mean",
    "Temperature_Reference_Mean_C",
    "RH_Setpoint",
    "Temperature_Setpoint_C",
    "Day_ID",
    "Run_ID",
    "Test_ID",
}

assert set(deployable_predictors).isdisjoint(prohibited_predictor_columns)
assert target_column not in deployable_predictors
assert group_column not in deployable_predictors

X_deployable = calibration_summary_df[deployable_predictors].copy()
y_correction = calibration_summary_df[target_column].copy()
cv_groups = calibration_summary_df[group_column].copy()

modeling_table_df = calibration_summary_df[
    [
        "Test_ID",
        group_column,
        *deployable_predictors,
        target_column,
        *diagnostic_columns,
    ]
].copy()

assert len(X_deployable) == 96
assert len(y_correction) == 96
assert len(cv_groups) == 96
assert X_deployable.isna().sum().sum() == 0
assert y_correction.isna().sum() == 0
assert cv_groups.nunique() == 24

print("Deployable predictors :", deployable_predictors)
print("Target                :", target_column)
print("CV grouping variable  :", group_column)
print("Modeling rows         :", len(modeling_table_df))
print("Condition groups      :", cv_groups.nunique())

display(modeling_table_df.head(12).round(4))

print("\nCM-04 predictor-leakage audit: PASS")

## 9. Initial Curated Evidence Export

At this stage only the calibration-design summary is exported.

No model-comparison table, fitted coefficients, or validation result exists yet.

In [ ]:
# ------------------------------------------------------------
# Export initial HV-006 design summary
# ------------------------------------------------------------

design_summary_export = pd.DataFrame(
    [
        {"Item": "Measurement rows", "Value": len(calibration_df)},
        {"Item": "Calibration Test_IDs", "Value": design["N_Test_IDs"]},
        {"Item": "RH levels", "Value": ", ".join(map(str, design["RH_Levels"]))},
        {
            "Item": "Temperature levels C",
            "Value": ", ".join(map(str, design["Temperature_Levels_C"])),
        },
        {"Item": "Directions", "Value": ", ".join(design["Directions"])},
        {"Item": "Days", "Value": ", ".join(map(str, design["Days"]))},
        {"Item": "Runs", "Value": ", ".join(map(str, design["Runs"]))},
        {
            "Item": "Condition groups",
            "Value": design["N_Condition_Groups"],
        },
        {
            "Item": "Samples per Test_ID",
            "Value": ", ".join(
                map(str, design["Samples_Per_Test_ID_Unique"])
            ),
        },
        {
            "Item": "Reserved validation Test_IDs",
            "Value": validation_matrix_df["Test_ID"].nunique(),
        },
    ]
)

design_summary_path = (
    tables_dir / "hv006_calibration_design_summary.csv"
)

design_summary_export.to_csv(design_summary_path, index=False)

display(design_summary_export)
print(f"\nExported: {design_summary_path}")

## 10. Initial HV-006 Checkpoint

At this point the notebook should have established:

- a CALIBRATION-only working measurement frame;
- 5,760 calibration measurement rows;
- 96 independent Test_ID-level modeling records;
- 24 RH × Temperature × Direction condition groups;
- 60 samples per Test_ID;
- a verified correction-target sign convention;
- deployable predictor definitions;
- an explicit predictor-leakage barrier;
- no fitted model yet.

**Do not proceed to C0-C5 model fitting until Sections 4-9 reproduce successfully from a clean kernel.**

The next implementation stage will introduce the uncalibrated baseline (C0), constant-offset baseline (C1), and the group-aware cross-validation infrastructure before fitting more complex calibration models.